llguidance's core is a Rust extension (`llguidance/_lib`). Given a grammar (a JSON schema, a regular expression or a Lark grammar) and a tokenizer, it says at every step which tokens may come next. The examples use the built-in byte tokenizer (one token per byte), so no model, no GPU and no network are needed.

## Example 1 — the import, the version and the built-in byte tokenizer

In [ ]:
import importlib.metadata
import llguidance
from llguidance import LLTokenizer, _lib

print("llguidance", llguidance.__version__, "--", llguidance.get_version())
assert llguidance.__version__ == importlib.metadata.version("llguidance"), (llguidance.__version__, importlib.metadata.version("llguidance"))
assert llguidance.get_version().startswith("llguidance@" + llguidance.__version__), llguidance.get_version()
assert _lib.__file__.endswith(".so"), _lib.__file__
tokenizer = LLTokenizer("byte")  # 256 byte tokens plus a few special ones
assert tokenizer.vocab_size >= 257 and tokenizer.is_special_token(tokenizer.eos_token)
tokens = tokenizer.tokenize_str("hi{")
assert tokens == [104, 105, 123], tokens  # one token per byte: h, i, {
assert tokenizer.decode_str(tokenizer.tokenize_str("héllo")) == "héllo"  # two bytes for é, and back
assert tokenizer.decode_bytes(tokens) == b"hi{"
print(f"OK -- byte tokenizer: {tokenizer.vocab_size} tokens, end token {tokenizer.eos_token}")

## Example 2 — a JSON schema: a valid answer is accepted, a wrong type stops at the bad token

In [ ]:
from llguidance import LLMatcher, LLTokenizer

tokenizer = LLTokenizer("byte")
schema = {"type": "object", "properties": {"name": {"type": "string"}, "age": {"type": "integer"}},
          "required": ["name", "age"], "additionalProperties": False}
grammar = LLMatcher.grammar_from_json_schema(schema)
assert LLMatcher.validate_grammar(grammar, tokenizer) == "", LLMatcher.validate_grammar(grammar, tokenizer)
matcher = LLMatcher(tokenizer, grammar)
assert not matcher.is_error(), matcher.get_error()
good = tokenizer.tokenize_str('{"name":"Ada","age":36}')
assert matcher.validate_tokens(good) == len(good)  # how many leading tokens fit: all of them
assert matcher.try_consume_tokens(good) == len(good) and matcher.is_accepting()
matcher.reset()
bad = tokenizer.tokenize_str('{"name":5')  # a number where the schema wants a string
fitting = matcher.validate_tokens(bad)
assert fitting == len(bad) - 1, (fitting, len(bad))  # everything fits except the last token, the 5
assert not matcher.is_error()  # validating does not change the matcher
print(f"OK -- {len(good)} tokens of a valid answer accepted; the wrong-type answer stops at token {fitting}: '5'")

## Example 3 — regular expressions and Lark grammars, and text the grammar forces

In [ ]:
import json
from llguidance import LLMatcher, LLTokenizer, grammar_from, regex_to_lark
from llguidance.gbnf_to_lark import gbnf_to_lark

tokenizer = LLTokenizer("byte")
phone = LLMatcher(tokenizer, LLMatcher.grammar_from_regex(r"\d{3}-\d{4}"))
assert phone.try_consume_tokens(tokenizer.tokenize_str("555-1234")) == 8 and phone.is_accepting()
choice = LLMatcher(tokenizer, LLMatcher.grammar_from_lark('start: "yes" | "no"'))
assert choice.try_consume_tokens(tokenizer.tokenize_str("no")) == 2 and choice.is_accepting()
integer = LLMatcher(tokenizer, grammar_from("json_schema", json.dumps({"type": "integer"})))
assert integer.try_consume_tokens(tokenizer.tokenize_str("42")) == 2 and integer.is_accepting()
forced = LLMatcher(tokenizer, LLMatcher.grammar_from_lark('start: "hello world" /[0-9]+/'))
assert forced.compute_ff_bytes() == b"hello world"  # the only possible start: no need to ask the model
forced.consume_tokens(forced.compute_ff_tokens())
assert not forced.is_accepting() and forced.compute_ff_bytes() == b""  # digits are needed, and they are free
forced.consume_tokens(tokenizer.tokenize_str("7"))
assert forced.is_accepting()
assert 'start: "a"' in gbnf_to_lark('root ::= "a" | "b"\n') and regex_to_lark("a/b") == "a\\/b"
print("OK -- regex, Lark and JSON grammars each accept their text; a fixed prefix is fast-forwarded:", b"hello world")

## Example 4 — the allowed-token bitmask, rollback and independent copies

In [ ]:
import numpy as np
import llguidance.numpy as llg_numpy
from llguidance import LLMatcher, LLTokenizer

tokenizer = LLTokenizer("byte")
matcher = LLMatcher(tokenizer, LLMatcher.grammar_from_lark('start: "id=" /[0-9]+/'))
matcher.consume_tokens(tokenizer.tokenize_str("id="))
bitmask = np.zeros(llg_numpy.get_bitmask_shape(1, tokenizer.vocab_size), dtype=np.int32)
llg_numpy.fill_next_token_bitmask(matcher, bitmask, 0)  # one bit per token, 32 tokens per word
allowed = [t for t in range(tokenizer.vocab_size) if bitmask[0, t // 32] >> (t % 32) & 1]
assert sorted(tokenizer.decode_str([t]) for t in allowed) == list("0123456789"), allowed
raw = matcher.compute_bitmask()  # the same mask as bytes
assert [t for t in range(tokenizer.vocab_size) if raw[t // 8] >> (t % 8) & 1] == allowed
matcher.consume_tokens(tokenizer.tokenize_str("42"))
assert matcher.is_accepting()
matcher.rollback(2)  # take the two digits back
assert not matcher.is_accepting() and not matcher.is_error()
copy = matcher.deep_copy()
copy.consume_tokens(tokenizer.tokenize_str("7"))
assert copy.is_accepting() and not matcher.is_accepting()  # the copy moved, the original did not
print("OK -- after 'id=' exactly the ten digit tokens are allowed; rollback and deep_copy keep state apart")

## Example 5 — a broken grammar is reported as text, never a crash

In [ ]:
from llguidance import LLMatcher, LLTokenizer

tokenizer = LLTokenizer("byte")
report = LLMatcher.validate_grammar("start: undefined_rule", tokenizer)
assert "unknown name" in report and "undefined_rule" in report, report
broken = LLMatcher(tokenizer, "start: undefined_rule")  # building it does not raise ...
assert broken.is_error() and "undefined_rule" in broken.get_error()  # ... the matcher carries the error
bad_schema = LLMatcher.validate_grammar(LLMatcher.grammar_from_json_schema({"type": "no-such-type"}), tokenizer)
assert bad_schema != "", "an impossible JSON schema type was accepted"
strict = LLMatcher(tokenizer, LLMatcher.grammar_from_lark('start: "ab"'))
assert strict.try_consume_tokens(tokenizer.tokenize_str("ax")) == 1  # "a" fits, "x" does not
assert not strict.is_error() and not strict.is_accepting()  # trying is not an error; "b" is still owed
assert strict.try_consume_tokens(tokenizer.tokenize_str("b")) == 1 and strict.is_accepting()
print("OK -- an unknown rule and an impossible schema are reported:", report.splitlines()[0])